# Parallel Requests with panzer 2.2.0

BinPan 0.9.0 integrates panzer 2.2.0, which provides parallel request capabilities for the Binance API.
This notebook demonstrates:

1. **Automatic parallelization** — kline and aggTrade fetching is now parallelized transparently
2. **Bulk methods** — fetch data for multiple symbols in a single call
3. **Performance comparison** — sequential vs parallel timing

In [1]:
import binpan
from binpan.api.market import get_bulk_klines, get_bulk_depth
import pandas as pd
import time

binpan.__version__

'0.9.0'

## 1. Automatic Parallelization of Klines

When requesting more than 1000 candles, BinPan automatically splits the request into chunks and fetches them in parallel using `parallel_get`. This is completely transparent — the API is the same.

In [2]:
# 2500 candles of 1m = 3 parallel requests of 1000
start = time.time()
s = binpan.Symbol('BTCUSDT', '1m', limit=2500)
elapsed = time.time() - start

print(f"Fetched {len(s.df)} candles in {elapsed:.2f}s (parallel)")
s.df.head(3)

2026-03-06	 20:25:39     INFO get_candles_by_time_stamps -> symbol=BTCUSDT tick_interval=1m start=2026-03-05 02:44:00 end=2026-03-06 20:24:59 limit=2500


2026-03-06 20:25:39     INFO [panzer.binance.public.spot] Inicializando BinancePublicClient(market=spot)


2026-03-06 20:25:40     INFO [panzer.binance.public.spot] Rate limiter inicializado: max_per_minute=6000 safety_ratio=0.90


2026-03-06 20:25:41     INFO [panzer.binance.public.spot] Reloj sincronizado: offset=-114 ms


2026-03-06	 20:25:41     INFO Lanzando 3 peticiones de klines en paralelo para BTCUSDT


2026-03-06 20:25:41     INFO [panzer.binance.public.spot] parallel_get: 3 jobs, 1 lotes, used_local=29


Fetched 2501 candles in 3.23s (parallel)


,Open time,Open,High,Low,Close,Volume,Close time,Quote volume,Trades,Taker buy base volume,Taker buy quote volume,Ignore,Open timestamp,Close timestamp
BTCUSDT 1m Europe/Madrid,,,,,,,,,,,,,,
2026-03-05 02:44:00+01:00,2026-03-05 02:44:00+01:00,72699.99,72709.52,72605.72,72651.07,10.70676,2026-03-05 02:44:59.999000+01:00,7.777355e+05,4976,3.32277,241360.108485,0,1772675040000,1772675099999
2026-03-05 02:45:00+01:00,2026-03-05 02:45:00+01:00,72651.07,72666.00,72577.77,72625.33,13.77851,2026-03-05 02:45:59.999000+01:00,1.000582e+06,7241,5.75978,418292.349900,0,1772675100000,1772675159999
2026-03-05 02:46:00+01:00,2026-03-05 02:46:00+01:00,72625.34,72672.46,72625.34,72635.29,9.67922,2026-03-05 02:46:59.999000+01:00,7.031721e+05,5222,2.40811,174943.437209,0,1772675160000,1772675219999


In [3]:
# For comparison: 50 candles = single request, no parallelization overhead
start = time.time()
s2 = binpan.Symbol('BTCUSDT', '1m', limit=50)
elapsed = time.time() - start

print(f"Fetched {len(s2.df)} candles in {elapsed:.2f}s (single request)")

2026-03-06	 20:25:42     INFO get_candles_by_time_stamps -> symbol=BTCUSDT tick_interval=1m start=2026-03-06 19:34:00 end=2026-03-06 20:24:59 limit=50


Fetched 50 candles in 1.08s (single request)


## 2. Automatic Parallelization of Aggregated Trades

When fetching aggregated trades across multiple hours, each 1-hour chunk is now fetched in parallel. Sub-pagination within a chunk (when >1000 trades per hour) remains sequential as it depends on trade IDs.

In [4]:
ltc = binpan.Symbol('LTCUSDT', '15m', limit=50)

start = time.time()
ltc.get_agg_trades(hours=4)
elapsed = time.time() - start

print(f"Fetched {len(ltc.agg_trades)} aggregated trades in {elapsed:.2f}s (parallel hourly chunks)")
ltc.agg_trades.head(3)

2026-03-06	 20:25:43     INFO get_candles_by_time_stamps -> symbol=LTCUSDT tick_interval=15m start=2026-03-06 07:30:00 end=2026-03-06 20:14:59 limit=50


2026-03-06	 20:25:44     INFO Obteniendo aggTrades de LTCUSDT por tiempo: 2026-03-06 15:25:44.684000 -> 2026-03-06 18:45:00.000000


2026-03-06	 20:25:44     INFO Lanzando 4 peticiones de aggTrades en paralelo para LTCUSDT


Requesting aggregated trades between 2026-03-06 16:25:44 and 2026-03-06 19:45:00


2026-03-06 20:25:45     INFO [panzer.binance.public.spot] parallel_get: 4 jobs, 1 lotes, used_local=109


2026-03-06	 20:25:47     INFO aggTrades LTCUSDT: 13 peticiones, 10823 trades obtenidos


Fetched 10823 aggregated trades in 3.02s (parallel hourly chunks)


,Aggregate tradeId,Price,Quantity,First tradeId,Last tradeId,Date,Timestamp,Buyer was maker,Best price match
LTCUSDT Europe/Madrid,,,,,,,,,
0,324356205,53.93,0.352,566820486,566820487,2026-03-06 16:25:45,1772810745026,False,True
1,324356206,53.93,0.769,566820488,566820488,2026-03-06 16:25:45,1772810745849,False,True
2,324356207,53.93,1.841,566820489,566820490,2026-03-06 16:25:47,1772810747047,False,True


## 3. Bulk Klines — Multiple Symbols in Parallel

The new `get_bulk_klines()` function fetches klines for multiple symbols simultaneously using panzer's `bulk_klines`. This is ideal for screeners, dashboards, or any multi-symbol analysis.

Returns a dictionary mapping each symbol to its kline data.

In [5]:
symbols = ['BTCUSDT', 'ETHUSDT', 'LTCUSDT', 'SOLUSDT', 'DOTUSDT',
           'ADAUSDT', 'XRPUSDT', 'BNBUSDT', 'DOGEUSDT', 'AVAXUSDT']

start = time.time()
bulk_data = get_bulk_klines(symbols, '15m', limit=100)
elapsed = time.time() - start

print(f"Fetched klines for {len(bulk_data)} symbols in {elapsed:.2f}s\n")
for sym, klines in bulk_data.items():
    print(f"  {sym}: {len(klines)} candles")

2026-03-06	 20:25:47     INFO bulk_klines: 10 símbolos, interval=15m, limit=100


2026-03-06 20:25:48     INFO [panzer.binance.public.spot] parallel_get: 10 jobs, 1 lotes, used_local=165


Fetched klines for 10 symbols in 0.36s

  BTCUSDT: 100 candles
  ETHUSDT: 100 candles
  LTCUSDT: 100 candles
  SOLUSDT: 100 candles
  DOTUSDT: 100 candles
  ADAUSDT: 100 candles
  XRPUSDT: 100 candles
  BNBUSDT: 100 candles
  DOGEUSDT: 100 candles
  AVAXUSDT: 100 candles


In [6]:
# Convert bulk data to DataFrames for analysis
from binpan.api.market import parse_candles_to_dataframe

dfs = {}
for sym, klines in bulk_data.items():
    dfs[sym] = parse_candles_to_dataframe(klines, symbol=sym, tick_interval='15m', time_zone='Europe/Madrid')

# Show last close prices
last_prices = {sym: df['Close'].iloc[-1] for sym, df in dfs.items()}
pd.Series(last_prices, name='Last Close').to_frame()

,Last Close
BTCUSDT,68181.93000
ETHUSDT,1983.78000
LTCUSDT,53.60000
SOLUSDT,85.51000
DOTUSDT,1.48600
ADAUSDT,0.25960
XRPUSDT,1.36170
BNBUSDT,630.84000
DOGEUSDT,0.09082
AVAXUSDT,8.95000


## 4. Bulk Depth — Multiple Order Books in Parallel

`get_bulk_depth()` fetches order book snapshots for multiple symbols simultaneously.

In [7]:
depth_symbols = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'XRPUSDT', 'BNBUSDT']

start = time.time()
depth_data = get_bulk_depth(depth_symbols, limit=20)
elapsed = time.time() - start

print(f"Fetched depth for {len(depth_data)} symbols in {elapsed:.2f}s\n")
for sym, book in depth_data.items():
    best_bid = float(book['bids'][0][0])
    best_ask = float(book['asks'][0][0])
    spread_bps = (best_ask - best_bid) / best_bid * 10000
    print(f"  {sym}: bid={best_bid:.4f}  ask={best_ask:.4f}  spread={spread_bps:.2f} bps")

2026-03-06	 20:25:48     INFO bulk_depth: 5 símbolos, limit=20


2026-03-06 20:25:48     INFO [panzer.binance.public.spot] parallel_get: 5 jobs, 1 lotes, used_local=190


Fetched depth for 5 symbols in 0.33s

  BTCUSDT: bid=68181.9300  ask=68181.9400  spread=0.00 bps
  ETHUSDT: bid=1983.7700  ask=1983.7800  spread=0.05 bps
  SOLUSDT: bid=85.5100  ask=85.5200  spread=1.17 bps
  XRPUSDT: bid=1.3616  ask=1.3617  spread=0.73 bps
  BNBUSDT: bid=630.8400  ask=630.8500  spread=0.16 bps


## 5. Practical Example: Multi-Symbol EMA Screener

Combine bulk klines with technical analysis to quickly scan multiple symbols.

In [8]:
import numpy as np
from binpan.analysis.numba_tools import ema_numba

screener_symbols = ['BTCUSDT', 'ETHUSDT', 'LTCUSDT', 'SOLUSDT', 'DOTUSDT',
                     'ADAUSDT', 'XRPUSDT', 'BNBUSDT', 'DOGEUSDT', 'AVAXUSDT']

# Fetch 200 candles for all symbols at once
start = time.time()
bulk = get_bulk_klines(screener_symbols, '1h', limit=200)
elapsed_fetch = time.time() - start

# Calculate EMA crossover for each
results = []
for sym, klines in bulk.items():
    df = parse_candles_to_dataframe(klines, symbol=sym, tick_interval='1h')
    if len(df) < 50:
        continue
    closes = df['Close'].to_numpy(dtype=np.float64)
    ema_fast = ema_numba(closes, 21)
    ema_slow = ema_numba(closes, 50)
    last_close = df['Close'].iloc[-1]
    fast_above = ema_fast[-1] > ema_slow[-1]
    results.append({
        'Symbol': sym,
        'Price': last_close,
        'EMA21': round(ema_fast[-1], 4),
        'EMA50': round(ema_slow[-1], 4),
        'Trend': 'Bullish' if fast_above else 'Bearish'
    })

screener_df = pd.DataFrame(results).set_index('Symbol')
elapsed_total = time.time() - start

print(f"Bulk fetch: {elapsed_fetch:.2f}s | Total (fetch + EMA): {elapsed_total:.2f}s\n")
screener_df

2026-03-06	 20:25:48     INFO bulk_klines: 10 símbolos, interval=1h, limit=200


2026-03-06 20:25:48     INFO [panzer.binance.public.spot] parallel_get: 10 jobs, 1 lotes, used_local=210


/home/nando/PycharmProjects/binpan_studio_dev/binpan/api/market.py:403: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['2026-02-26 12:00:00.000000' '2026-02-26 13:00:00.000000'
 '2026-02-26 14:00:00.000000' '2026-02-26 15:00:00.000000'
 '2026-02-26 16:00:00.000000' '2026-02-26 17:00:00.000000'
 '2026-02-26 18:00:00.000000' '2026-02-26 19:00:00.000000'
 '2026-02-26 20:00:00.000000' '2026-02-26 21:00:00.000000'
 '2026-02-26 22:00:00.000000' '2026-02-26 23:00:00.000000'
 '2026-02-27 00:00:00.000000' '2026-02-27 01:00:00.000000'
 '2026-02-27 02:00:00.000000' '2026-02-27 03:00:00.000000'
 '2026-02-27 04:00:00.000000' '2026-02-27 05:00:00.000000'
 '2026-02-27 06:00:00.000000' '2026-02-27 07:00:00.000000'
 '2026-02-27 08:00:00.000000' '2026-02-27 09:00:00.000000'
 '2026-02-27 10:00:00.000000' '2026-02-27 11:00:00.000000'
 '2026-02-27 12:00:00.000000' '2026-02-27 13:00:00.000000'
 '2026-02-27 14:00:00.000000' '2026-02-27 1

/home/nando/PycharmProjects/binpan_studio_dev/binpan/api/market.py:403: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['2026-02-26 12:00:00.000000' '2026-02-26 13:00:00.000000'
 '2026-02-26 14:00:00.000000' '2026-02-26 15:00:00.000000'
 '2026-02-26 16:00:00.000000' '2026-02-26 17:00:00.000000'
 '2026-02-26 18:00:00.000000' '2026-02-26 19:00:00.000000'
 '2026-02-26 20:00:00.000000' '2026-02-26 21:00:00.000000'
 '2026-02-26 22:00:00.000000' '2026-02-26 23:00:00.000000'
 '2026-02-27 00:00:00.000000' '2026-02-27 01:00:00.000000'
 '2026-02-27 02:00:00.000000' '2026-02-27 03:00:00.000000'
 '2026-02-27 04:00:00.000000' '2026-02-27 05:00:00.000000'
 '2026-02-27 06:00:00.000000' '2026-02-27 07:00:00.000000'
 '2026-02-27 08:00:00.000000' '2026-02-27 09:00:00.000000'
 '2026-02-27 10:00:00.000000' '2026-02-27 11:00:00.000000'
 '2026-02-27 12:00:00.000000' '2026-02-27 13:00:00.000000'
 '2026-02-27 14:00:00.000000' '2026-02-27 1

Bulk fetch: 0.37s | Total (fetch + EMA): 0.64s



/home/nando/PycharmProjects/binpan_studio_dev/binpan/api/market.py:403: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['2026-02-26 12:59:59.999000' '2026-02-26 13:59:59.999000'
 '2026-02-26 14:59:59.999000' '2026-02-26 15:59:59.999000'
 '2026-02-26 16:59:59.999000' '2026-02-26 17:59:59.999000'
 '2026-02-26 18:59:59.999000' '2026-02-26 19:59:59.999000'
 '2026-02-26 20:59:59.999000' '2026-02-26 21:59:59.999000'
 '2026-02-26 22:59:59.999000' '2026-02-26 23:59:59.999000'
 '2026-02-27 00:59:59.999000' '2026-02-27 01:59:59.999000'
 '2026-02-27 02:59:59.999000' '2026-02-27 03:59:59.999000'
 '2026-02-27 04:59:59.999000' '2026-02-27 05:59:59.999000'
 '2026-02-27 06:59:59.999000' '2026-02-27 07:59:59.999000'
 '2026-02-27 08:59:59.999000' '2026-02-27 09:59:59.999000'
 '2026-02-27 10:59:59.999000' '2026-02-27 11:59:59.999000'
 '2026-02-27 12:59:59.999000' '2026-02-27 13:59:59.999000'
 '2026-02-27 14:59:59.999000' '2026-02-27 1

,Price,EMA21,EMA50,Trend
Symbol,,,,
BTCUSDT,68181.93000,69667.2908,70341.8835,Bearish
ETHUSDT,1983.77000,2029.4236,2053.2119,Bearish
LTCUSDT,53.60000,54.5642,55.1210,Bearish
SOLUSDT,85.51000,86.6863,87.8123,Bearish
DOTUSDT,1.48600,1.5001,1.5136,Bearish
ADAUSDT,0.25960,0.2642,0.2677,Bearish
XRPUSDT,1.36170,1.3824,1.3949,Bearish
BNBUSDT,630.84000,637.8795,642.7534,Bearish
DOGEUSDT,0.09082,0.0923,0.0934,Bearish


## Summary

| Feature | Method | Use Case |
|---------|--------|----------|
| Auto-parallel klines | `Symbol('X', '1m', limit=5000)` | Large candle ranges (transparent) |
| Auto-parallel aggTrades | `symbol.get_agg_trades(hours=N)` | Multi-hour trade fetching (transparent) |
| Bulk klines | `get_bulk_klines(symbols, interval)` | Screeners, multi-symbol dashboards |
| Bulk depth | `get_bulk_depth(symbols)` | Order book snapshots for multiple pairs |

All parallelization respects panzer's rate limiter — no risk of hitting API limits.